In [ ]:
import sys
import os
import importlib
import time

importlib.invalidate_caches()

target_path = '/content/drive/MyDrive/Thesis/models'
if target_path not in sys.path:
    sys.path.insert(0, target_path)

import argparse
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
import json

from utils import (
    DATA_DIR, MODELS_DIR, GOEMOTIONS_LABELS, RESULTS_DIR,
    EKMAN_LABELS, EKMAN_LABEL_TO_ID, set_all_seeds,
)
from evaluate import compute_metrics, save_metrics, plot_confusion_matrix


In [ ]:
def load_partitions(preprocessing: str = "heavy"):
    assert preprocessing in ("light", "heavy")

    suffix = "clean" if preprocessing == "heavy" else "light"


    train = pd.read_parquet(DATA_DIR / f"train_single_{suffix}.parquet")
    val   = pd.read_parquet(DATA_DIR / f"validation_single_{suffix}.parquet")
    test  = pd.read_parquet(DATA_DIR / f"test_single_{suffix}.parquet")
    return train, val, test

In [ ]:
def make_pipeline() -> Pipeline:
    return Pipeline([
        ("tfidf", TfidfVectorizer(max_features=20_000,
                                   ngram_range=(1, 2),
                                   min_df=2)),
        ("clf",   LogisticRegression(class_weight="balanced",
                                      max_iter=1000,
                                      C=1.0,
                                      multi_class="multinomial",
                                      solver="lbfgs",
                                      n_jobs=-1)),
    ])

In [ ]:
def run(phase: int, seed: int = 42, preprocessing: str = "heavy") -> None:
    assert phase in (1, 2)
    assert preprocessing in ("light", "heavy")

    if phase == 1 and preprocessing == "light":
      raise ValueError("The light preprocessing ablation is for phase 2!")

    set_all_seeds(seed)
    train, _, test = load_partitions(preprocessing=preprocessing)

    if phase == 1:
        run_name = f"phase1_lr_28class_seed{seed}"
        labels   = GOEMOTIONS_LABELS
        y_train  = train["label"].values
        y_test   = test["label"].values
    else:
        run_name = f"phase2_lr_ekman7_seed{seed}"
        labels   = EKMAN_LABELS
        y_train  = train["ekman_id"].values
        y_test   = test["ekman_id"].values

        if preprocessing == "heavy":
          run_name = f"phase2_lr_ekman7_heavy_seed{seed}"
        else:
          run_name = f"phase2_lr_ekman7_light_seed{seed}"


    pipe = make_pipeline()

    print(f"[{run_name}] Preprocessing = {preprocessing}")
    print(f"[{run_name}] Training on {len(train):,} samples...")
    print(f"[{run_name}] Starting training...")

    _t0 = time.time()
    pipe.fit(train["text_clean"].values, y_train)
    _elapsed = time.time() - _t0
    print(f"[{run_name}] TRAINING TIME = {_elapsed:.1f} seconds")

    print(f"[{run_name}] Evaluating on {len(test):,} samples...")
    y_pred  = pipe.predict(test["text_clean"].values)

    metrics = compute_metrics(y_test, y_pred, labels)
    save_metrics(metrics, run_name)
    plot_confusion_matrix(y_test, y_pred, labels, run_name)

    out = MODELS_DIR / f"{run_name}.joblib"
    joblib.dump(pipe, out)
    print(f"[{run_name}] Saved model -> {out}")
    print(f"[{run_name}] Macro-F1   = {metrics['macro_f1']:.4f}")
    print(f"[{run_name}] Weighted-F1= {metrics['weighted_f1']:.4f}")
    print(f"[{run_name}] Accuracy   = {metrics['accuracy']:.4f}")


In [ ]:
run(phase=1, seed=42,preprocessing= "heavy")

KeyboardInterrupt: 

In [ ]:
run(phase=2, seed=42, preprocessing="heavy")

[phase2_lr_ekman7_heavy_seed42] Preprocessing = heavy
[phase2_lr_ekman7_heavy_seed42] Training on 36,117 samples...
[phase2_lr_ekman7_heavy_seed42] Starting training...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[phase2_lr_ekman7_heavy_seed42] TRAINING TIME = 7.7 seconds
[phase2_lr_ekman7_heavy_seed42] Evaluating on 4,568 samples...
Saved metrics -> /content/drive/MyDrive/Thesis/results/phase2_lr_ekman7_heavy_seed42.json
Saved figure -> /content/drive/MyDrive/Thesis/figures/cm_phase2_lr_ekman7_heavy_seed42.png
[phase2_lr_ekman7_heavy_seed42] Saved model -> /content/drive/MyDrive/Thesis/models/phase2_lr_ekman7_heavy_seed42.joblib
[phase2_lr_ekman7_heavy_seed42] Macro-F1   = 0.5145
[phase2_lr_ekman7_heavy_seed42] Weighted-F1= 0.5863
[phase2_lr_ekman7_heavy_seed42] Accuracy   = 0.5749


In [ ]:
run(phase=2, seed=42, preprocessing="light")

[phase2_lr_ekman7_light_seed42] Preprocessing = light
[phase2_lr_ekman7_light_seed42] Training on 36,302 samples...
[phase2_lr_ekman7_light_seed42] Starting training...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


[phase2_lr_ekman7_light_seed42] TRAINING TIME = 6.4 seconds
[phase2_lr_ekman7_light_seed42] Evaluating on 4,590 samples...
Saved metrics -> /content/drive/MyDrive/Thesis/results/phase2_lr_ekman7_light_seed42.json
Saved figure -> /content/drive/MyDrive/Thesis/figures/cm_phase2_lr_ekman7_light_seed42.png
[phase2_lr_ekman7_light_seed42] Saved model -> /content/drive/MyDrive/Thesis/models/phase2_lr_ekman7_light_seed42.joblib
[phase2_lr_ekman7_light_seed42] Macro-F1   = 0.5363
[phase2_lr_ekman7_light_seed42] Weighted-F1= 0.6065
[phase2_lr_ekman7_light_seed42] Accuracy   = 0.5952


In [ ]:
run_names =[
    "phase2_lr_ekman7_heavy_seed42",
    "phase2_lr_ekman7_light_seed42"
]

rows = []
for name in run_names:
  path = RESULTS_DIR / f"{name}.json"
  if path.exists():
    with open(path) as f:
      m= json.load(f)
    rows.append({
        "run": name,
        "macro_f1": m["macro_f1"],
        "weighted_f1": m["weighted_f1"],
        "accuracy": m["accuracy"],
    })
  else:
    print(f"Missing results file: {path}")

pd.DataFrame(rows)

,run,macro_f1,weighted_f1,accuracy
0,phase2_lr_ekman7_heavy_seed42,0.514482,0.586285,0.574869
1,phase2_lr_ekman7_light_seed42,0.536283,0.606545,0.595207
